In [1]:
import sys
print(sys.executable)

C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026\.venv\Scripts\python.exe


In [2]:
import pandas
import numpy
import rapidfuzz

print("Environment OK")

Environment OK


In [6]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import normalize_name, normalize_address
from src.features import feature_pair

DATA_DIR = Path(r"C:\Users\manas\Downloads\train")

print("Environment ready")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("Pandas:", pd.__version__)

Environment ready
ROOT: C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026
DATA_DIR: C:\Users\manas\Downloads\train
Pandas: 2.3.3


In [4]:
DATA_DIR = Path(r"C:\Users\manas\Downloads\train")

print("Data directory:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

for filename in [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
]:
    path = DATA_DIR / filename
    print(filename, "->", path.exists())

Data directory: C:\Users\manas\Downloads\train
Exists: True
train_source1.tsv -> True
train_source2.tsv -> True
train_source3.tsv -> True
train_ground_truth.tsv -> True


In [7]:
s1 = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

s2 = pd.read_csv(
    DATA_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

s3 = pd.read_csv(
    DATA_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (10000, 4)
S2: (10000, 4)
S3: (10000, 4)


In [8]:
print("S1 columns:")
print(s1.columns.tolist())

print("\nS2 columns:")
print(s2.columns.tolist())

print("\nS3 columns:")
print(s3.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [9]:
display(s1.head(3))
display(s2.head(3))
display(s3.head(3))

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US


In [10]:
gt = pd.read_csv(
    DATA_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    nrows=10_000,
    keep_default_na=False,
)

print("GT shape:", gt.shape)
print("GT columns:", gt.columns.tolist())

display(gt.head(10))

GT shape: (10000, 2)
GT columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [11]:
def prepare_source(df):
    df = df.copy()

    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)

    return df


s1_p = prepare_source(s1)
s2_p = prepare_source(s2)
s3_p = prepare_source(s3)

In [12]:
display(
    s1_p[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country",
        ]
    ].head(10)
)

,business_name,name_norm,business_address,address_norm,country
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US


In [13]:
s2_lookup = s2_p.set_index("entity_id").to_dict("index")
s3_lookup = s3_p.set_index("entity_id").to_dict("index")

In [14]:
s1_lookup = s1_p.set_index("entity_id").to_dict("index")

In [15]:
display(
    gt[
        ["source1_entity_id", "matched_entity_ids"]
    ].head(20)
)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [16]:
for _, row in gt.head(10).iterrows():
    print("S1:", row["source1_entity_id"])
    print("Matches:", row["matched_entity_ids"])
    print()

S1: S1-965667
Matches: S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364

S1: S1-55344266
Matches: S2-249013014,S2-197070651,S3-478195123,S3-384364074

S1: S1-343815751
Matches: S2-790675320,S2-479876582,S3-878454467

S1: S1-656753428
Matches: S2-153058913,S2-24659151,S3-679606215

S1: S1-102811957
Matches: S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785

S1: S1-18727616
Matches: S2-755677256,S3-187831601,S3-641489370,S3-476250621

S1: S1-318373630
Matches: S2-660036492,S3-804600254

S1: S1-86989137
Matches: S3-274817120,S3-312496301

S1: S1-29845983
Matches: S2-648035184,S3-588502663

S1: S1-789009573
Matches: S2-383871912,S3-74481402,S3-576451439



In [18]:
print("S1:", s1.columns.tolist())
print("S2:", s2.columns.tolist())
print("S3:", s3.columns.tolist())
print("GT:", gt.columns.tolist())

S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']


In [19]:
# Load ONLY entity IDs from the full training sources.
# This is much smaller than loading all business fields.

s1_ids = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

s2_ids = pd.read_csv(
    DATA_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

s3_ids = pd.read_csv(
    DATA_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    usecols=["entity_id"],
    keep_default_na=False,
)

print("S1 IDs:", len(s1_ids))
print("S2 IDs:", len(s2_ids))
print("S3 IDs:", len(s3_ids))

S1 IDs: 2206821
S2 IDs: 5034616
S3 IDs: 5285603


In [20]:
s1_id_set = set(s1_ids["entity_id"])
s2_id_set = set(s2_ids["entity_id"])
s3_id_set = set(s3_ids["entity_id"])

print("Unique S1 IDs:", len(s1_id_set))
print("Unique S2 IDs:", len(s2_id_set))
print("Unique S3 IDs:", len(s3_id_set))

Unique S1 IDs: 2206821
Unique S2 IDs: 5034616
Unique S3 IDs: 5285603


In [21]:
gt_full = pd.read_csv(
    DATA_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

print("GT rows:", len(gt_full))
display(gt_full.head())

GT rows: 2206821


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [23]:
gt_work = gt_full.copy()

gt_work = gt_work[
    gt_work["source1_entity_id"].isin(s1_id_set)
].copy()

gt_work = gt_work[
    gt_work["matched_entity_ids"].ne("")
].copy()

print("GT rows with S1 present and non-empty matches:", len(gt_work))

GT rows with S1 present and non-empty matches: 2083574


In [24]:
links = (
    gt_work[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x:
            x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

links["matched_entity_id"] = (
    links["matched_entity_id"]
    .str.strip()
)

print("Expanded GT links:", len(links))

Expanded GT links: 7638365


In [26]:
links["source"] = np.where(
    links["matched_entity_id"].isin(s2_id_set),
    "S2",
    np.where(
        links["matched_entity_id"].isin(s3_id_set),
        "S3",
        "UNAVAILABLE"
    )
)

In [27]:
positive_links = links[
    links["source"].isin(["S2", "S3"])
].copy()

positive_links = positive_links.rename(
    columns={
        "source1_entity_id": "s1_id",
        "matched_entity_id": "matched_id",
    }
)[
    ["s1_id", "matched_id", "source"]
]

print("Valid positive links:", len(positive_links))
display(positive_links.head(20))

Valid positive links: 7638365


,s1_id,matched_id,source
0,S1-965667,S2-681193310,S2
0,S1-965667,S2-743505751,S2
0,S1-965667,S3-775321672,S3
0,S1-965667,S3-11291185,S3
0,S1-965667,S3-860443364,S3
1,S1-55344266,S2-249013014,S2
1,S1-55344266,S2-197070651,S2
1,S1-55344266,S3-478195123,S3
1,S1-55344266,S3-384364074,S3
2,S1-343815751,S2-790675320,S2


In [28]:
print(
    positive_links["source"].value_counts()
)

source
S3    3944746
S2    3693619
Name: count, dtype: int64


In [29]:
print("S1 IDs:", len(s1_id_set))
print("S2 IDs:", len(s2_id_set))
print("S3 IDs:", len(s3_id_set))

print("\nExample S2 IDs:")
print(list(s2_id_set)[:10])

print("\nExample S3 IDs:")
print(list(s3_id_set)[:10])

print("\nExample GT matched IDs:")
print(
    links["matched_entity_id"].head(20).tolist()
)

S1 IDs: 2206821
S2 IDs: 5034616
S3 IDs: 5285603

Example S2 IDs:
['S2-191653094', 'S2-366343257', 'S2-466344706', 'S2-295373860', 'S2-202816394', 'S2-189988461', 'S2-386492007', 'S2-527133944', 'S2-2438671', 'S2-60062257']

Example S3 IDs:
['S3-176718142', 'S3-325324922', 'S3-663745657', 'S3-745297842', 'S3-252983914', 'S3-195979398', 'S3-733863105', 'S3-368685433', 'S3-26645713', 'S3-340886654']

Example GT matched IDs:
['S2-681193310', 'S2-743505751', 'S3-775321672', 'S3-11291185', 'S3-860443364', 'S2-249013014', 'S2-197070651', 'S3-478195123', 'S3-384364074', 'S2-790675320', 'S2-479876582', 'S3-878454467', 'S2-153058913', 'S2-24659151', 'S3-679606215', 'S2-478959098', 'S2-553508714', 'S2-625774905', 'S3-728090388', 'S3-928796641']


In [30]:
s2_mask = links["matched_entity_id"].isin(s2_id_set)
s3_mask = links["matched_entity_id"].isin(s3_id_set)

print("In S2:", s2_mask.sum())
print("In S3:", s3_mask.sum())
print("In neither:", (~(s2_mask | s3_mask)).sum())

In S2: 3693619
In S3: 3944746
In neither: 0


In [31]:
overlap = s2_id_set & s3_id_set

print("S2 IDs:", len(s2_id_set))
print("S3 IDs:", len(s3_id_set))
print("S2/S3 overlapping IDs:", len(overlap))

S2 IDs: 5034616
S3 IDs: 5285603
S2/S3 overlapping IDs: 0


In [32]:
positive_counts = (
    positive_links
    .groupby(["s1_id", "source"])
    .size()
    .unstack(fill_value=0)
)

positive_counts["total"] = positive_counts.sum(axis=1)

print("S1 entities with positive links:", len(positive_counts))
print("\nPositive links by source:")
print(positive_links["source"].value_counts())

print("\nMatches per S1:")
print(positive_counts["total"].describe())

print("\nS1 entities by number of matches:")
print(positive_counts["total"].value_counts().sort_index())

S1 entities with positive links: 2083574

Positive links by source:
source
S3    3944746
S2    3693619
Name: count, dtype: int64

Matches per S1:
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: total, dtype: float64

S1 entities by number of matches:
total
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [33]:
# Sample positive pairs for feature inspection
positive_sample = positive_links.sample(
    n=2000,
    random_state=42
).reset_index(drop=True)

positive_sample.head()

,s1_id,matched_id,source
0,S1-341462203,S2-219553948,S2
1,S1-783955316,S3-656241763,S3
2,S1-313221336,S3-872559960,S3
3,S1-516394603,S2-642842175,S2
4,S1-887045319,S2-612158384,S2


In [34]:
# Load only the rows needed for our sampled positive IDs

s1_needed = set(positive_sample["s1_id"])

s2_needed = set(
    positive_sample.loc[
        positive_sample["source"] == "S2",
        "matched_id"
    ]
)

s3_needed = set(
    positive_sample.loc[
        positive_sample["source"] == "S3",
        "matched_id"
    ]
)

print("S1 needed:", len(s1_needed))
print("S2 needed:", len(s2_needed))
print("S3 needed:", len(s3_needed))

S1 needed: 1998
S2 needed: 965
S3 needed: 1035


In [35]:
def load_rows_by_ids(path, needed_ids, chunksize=200_000):
    pieces = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        matched = chunk[chunk["entity_id"].isin(needed_ids)]

        if not matched.empty:
            pieces.append(matched)

        if sum(len(x) for x in pieces) >= len(needed_ids):
            break

    if not pieces:
        return pd.DataFrame()

    return pd.concat(pieces, ignore_index=True)


s1_sample = load_rows_by_ids(
    DATA_DIR / "train_source1.tsv",
    s1_needed
)

s2_sample = load_rows_by_ids(
    DATA_DIR / "train_source2.tsv",
    s2_needed
)

s3_sample = load_rows_by_ids(
    DATA_DIR / "train_source3.tsv",
    s3_needed
)

print("S1 loaded:", len(s1_sample))
print("S2 loaded:", len(s2_sample))
print("S3 loaded:", len(s3_sample))

S1 loaded: 1998
S2 loaded: 965
S3 loaded: 1035


In [36]:
def prepare_source(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].map(normalize_name)
    df["address_norm"] = df["business_address"].map(normalize_address)
    return df


s1_sample = prepare_source(s1_sample)
s2_sample = prepare_source(s2_sample)
s3_sample = prepare_source(s3_sample)

print(s1_sample[[
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "name_norm",
    "address_norm"
]].head())

      entity_id                        business_name  \
0    S1-6232617                      Aspen Ridge LLC   
1  S1-277346734                  Ridge Detailing Inc   
2  S1-951451939      Maa Investments Private Limited   
3  S1-411744587  Vinita Herrell Sterling Managed LLC   
4  S1-804210432    Keystone Creative Associates PLLC   

                                    business_address country  \
0             TX, 834 Sunshine Medley Lane, Richmond      US   
1                  Rumford, 660 Penobscot Street, ME      US   
2  Villa No 68, Cyprus Palms, Serilingampally, K....   India   
3  2201 Chesapeake Harbour Drive, Fl 0, Annapolis...      US   
4                    211 Woodway Avenue, Spokane, WA      US   

                             name_norm  \
0                      aspen ridge llc   
1                  ridge detailing inc   
2      maa investments private limited   
3  vinita herrell sterling managed llc   
4    keystone creative associates pllc   

                         

In [37]:
s1_lookup = s1_sample.set_index("entity_id").to_dict("index")
s2_lookup = s2_sample.set_index("entity_id").to_dict("index")
s3_lookup = s3_sample.set_index("entity_id").to_dict("index")

print("Lookup sizes:")
print("S1:", len(s1_lookup))
print("S2:", len(s2_lookup))
print("S3:", len(s3_lookup))

Lookup sizes:
S1: 1998
S2: 965
S3: 1035


In [38]:
feature_rows = []

for _, pair in positive_sample.iterrows():
    s1_row = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match_row = s2_lookup[pair["matched_id"]]
    else:
        match_row = s3_lookup[pair["matched_id"]]

    features = feature_pair(s1_row, match_row)

    features["s1_id"] = pair["s1_id"]
    features["matched_id"] = pair["matched_id"]
    features["source"] = pair["source"]
    features["label"] = 1

    feature_rows.append(features)

positive_features = pd.DataFrame(feature_rows)

print("Feature rows:", len(positive_features))
print("Feature columns:")
print(positive_features.columns.tolist())

Feature rows: 2000
Feature columns:
['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact', 's1_id', 'matched_id', 'source', 'label']


In [39]:
positive_features.head().T

,0,1,2,3,4
name_exact,0,1,1,0,0
name_ratio,0.21875,1.0,1.0,0.933333,0.948718
name_token_set,0.258065,1.0,1.0,0.933333,0.666667
name_token_sort,0.21875,1.0,1.0,0.933333,0.666667
name_jaccard,0.066667,1.0,1.0,0.6,0.428571
name_char_jaccard,0.054545,1.0,1.0,0.576923,0.731707
name_length_difference,2,0,0,1,2
name_length_ratio,0.939394,1.0,1.0,0.956522,0.95
address_exact,0,0,0,0,0
address_ratio,0.84,0.45,0.760563,0.765432,0.892308


In [40]:
positive_features.describe().T

,count,mean,std,min,25%,50%,75%,max
name_exact,2000.0,0.221000,0.415024,0.000000,0.000000,0.000000,0.000000,1.0
name_ratio,2000.0,0.804384,0.231938,0.064516,0.740487,0.875000,0.962963,1.0
name_token_set,2000.0,0.874260,0.241243,0.066667,0.887978,1.000000,1.000000,1.0
name_token_sort,2000.0,0.806790,0.241764,0.064516,0.738963,0.892857,1.000000,1.0
name_jaccard,2000.0,0.618253,0.329361,0.000000,0.500000,0.666667,1.000000,1.0
name_char_jaccard,2000.0,0.663982,0.285356,0.000000,0.500000,0.709005,0.878788,1.0
name_length_difference,2000.0,3.624000,4.896183,0.000000,0.000000,2.000000,4.000000,31.0
name_length_ratio,2000.0,0.866230,0.165903,0.095238,0.788534,0.928571,1.000000,1.0
address_exact,2000.0,0.089000,0.284815,0.000000,0.000000,0.000000,0.000000,1.0
address_ratio,2000.0,0.756729,0.232948,0.000000,0.642727,0.833333,0.921903,1.0


In [41]:
print("NaN values:")
print(positive_features.isna().sum())

print("\nInfinite values:")
print(np.isinf(
    positive_features.select_dtypes(include=np.number)
).sum().sum())

NaN values:
name_exact                   0
name_ratio                   0
name_token_set               0
name_token_sort              0
name_jaccard                 0
name_char_jaccard            0
name_length_difference       0
name_length_ratio            0
address_exact                0
address_ratio                0
address_token_set            0
address_token_sort           0
address_jaccard              0
address_char_jaccard         0
address_length_difference    0
address_length_ratio         0
digit_overlap                0
country_same                 0
both_exact                   0
s1_id                        0
matched_id                   0
source                       0
label                        0
dtype: int64

Infinite values:
0


In [43]:
print(positive_features.columns.tolist())
print("Number of feature columns:", len(positive_features.columns))
positive_features.head().T

['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact', 's1_id', 'matched_id', 'source', 'label']
Number of feature columns: 23


,0,1,2,3,4
name_exact,0,1,1,0,0
name_ratio,0.21875,1.0,1.0,0.933333,0.948718
name_token_set,0.258065,1.0,1.0,0.933333,0.666667
name_token_sort,0.21875,1.0,1.0,0.933333,0.666667
name_jaccard,0.066667,1.0,1.0,0.6,0.428571
name_char_jaccard,0.054545,1.0,1.0,0.576923,0.731707
name_length_difference,2,0,0,1,2
name_length_ratio,0.939394,1.0,1.0,0.956522,0.95
address_exact,0,0,0,0,0
address_ratio,0.84,0.45,0.760563,0.765432,0.892308


In [44]:
feature_cols = [
    col for col in positive_features.columns
    if col not in ["s1_id", "matched_id", "source", "label"]
]

print("Features being analyzed:")
print(feature_cols)

positive_features[feature_cols].describe().T

Features being analyzed:
['name_exact', 'name_ratio', 'name_token_set', 'name_token_sort', 'name_jaccard', 'name_char_jaccard', 'name_length_difference', 'name_length_ratio', 'address_exact', 'address_ratio', 'address_token_set', 'address_token_sort', 'address_jaccard', 'address_char_jaccard', 'address_length_difference', 'address_length_ratio', 'digit_overlap', 'country_same', 'both_exact']


,count,mean,std,min,25%,50%,75%,max
name_exact,2000.0,0.221000,0.415024,0.000000,0.000000,0.000000,0.000000,1.0
name_ratio,2000.0,0.804384,0.231938,0.064516,0.740487,0.875000,0.962963,1.0
name_token_set,2000.0,0.874260,0.241243,0.066667,0.887978,1.000000,1.000000,1.0
name_token_sort,2000.0,0.806790,0.241764,0.064516,0.738963,0.892857,1.000000,1.0
name_jaccard,2000.0,0.618253,0.329361,0.000000,0.500000,0.666667,1.000000,1.0
name_char_jaccard,2000.0,0.663982,0.285356,0.000000,0.500000,0.709005,0.878788,1.0
name_length_difference,2000.0,3.624000,4.896183,0.000000,0.000000,2.000000,4.000000,31.0
name_length_ratio,2000.0,0.866230,0.165903,0.095238,0.788534,0.928571,1.000000,1.0
address_exact,2000.0,0.089000,0.284815,0.000000,0.000000,0.000000,0.000000,1.0
address_ratio,2000.0,0.756729,0.232948,0.000000,0.642727,0.833333,0.921903,1.0


In [45]:
print("NaN values:")
print(positive_features[feature_cols].isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        positive_features[feature_cols]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

NaN values:
name_exact                   0
name_ratio                   0
name_token_set               0
name_token_sort              0
name_jaccard                 0
name_char_jaccard            0
name_length_difference       0
name_length_ratio            0
address_exact                0
address_ratio                0
address_token_set            0
address_token_sort           0
address_jaccard              0
address_char_jaccard         0
address_length_difference    0
address_length_ratio         0
digit_overlap                0
country_same                 0
both_exact                   0
dtype: int64

Infinite values:
0


In [47]:
# Only keep GT links for the S1s in our 2,000-pair experiment
sample_s1_ids = set(positive_sample["s1_id"])

sample_positive_links = positive_links[
    positive_links["s1_id"].isin(sample_s1_ids)
].copy()

print("Positive links for sampled S1s:", len(sample_positive_links))
print("Unique sampled S1s:", sample_positive_links["s1_id"].nunique())

Positive links for sampled S1s: 8702
Unique sampled S1s: 1998


In [48]:
positive_lookup = (
    sample_positive_links
    .groupby("s1_id")["matched_id"]
    .agg(set)
    .to_dict()
)

print("S1s in positive lookup:", len(positive_lookup))

S1s in positive lookup: 1998


In [49]:
rng = np.random.default_rng(42)

negative_rows = []

candidate_s2_ids = s2_sample["entity_id"].tolist()
candidate_s3_ids = s3_sample["entity_id"].tolist()

sample_s1_ids = positive_sample["s1_id"].unique()

for s1_id in sample_s1_ids:

    known_matches = positive_lookup.get(s1_id, set())

    attempts = 0

    while attempts < 10:
        attempts += 1

        if rng.random() < 0.5:
            matched_id = rng.choice(candidate_s2_ids)
            source = "S2"
        else:
            matched_id = rng.choice(candidate_s3_ids)
            source = "S3"

        if matched_id not in known_matches:
            negative_rows.append({
                "s1_id": s1_id,
                "matched_id": matched_id,
                "source": source,
                "label": 0
            })
            break

    if len(negative_rows) >= 2000:
        break

negative_sample = pd.DataFrame(negative_rows)

print("Negative pairs:", len(negative_sample))
print("\nSource distribution:")
print(negative_sample["source"].value_counts())

Negative pairs: 1998

Source distribution:
source
S2    1017
S3     981
Name: count, dtype: int64


In [50]:
negative_feature_rows = []

for _, pair in negative_sample.iterrows():

    s1_row = s1_lookup[pair["s1_id"]]

    if pair["source"] == "S2":
        match_row = s2_lookup[pair["matched_id"]]
    else:
        match_row = s3_lookup[pair["matched_id"]]

    features = feature_pair(s1_row, match_row)

    features["s1_id"] = pair["s1_id"]
    features["matched_id"] = pair["matched_id"]
    features["source"] = pair["source"]
    features["label"] = 0

    negative_feature_rows.append(features)

negative_features = pd.DataFrame(negative_feature_rows)

print("Negative feature rows:", len(negative_features))

Negative feature rows: 1998


In [51]:
training_sample = pd.concat(
    [positive_features, negative_features],
    ignore_index=True
)

print("Total pairs:", len(training_sample))
print("\nLabel distribution:")
print(training_sample["label"].value_counts())

print("\nSource distribution:")
print(training_sample["source"].value_counts())

Total pairs: 3998

Label distribution:
label
1    2000
0    1998
Name: count, dtype: int64

Source distribution:
source
S3    2016
S2    1982
Name: count, dtype: int64


In [52]:
negative_check = negative_sample.apply(
    lambda row: row["matched_id"] in positive_lookup.get(row["s1_id"], set()),
    axis=1
)

print("Negative pairs accidentally known as positives:", negative_check.sum())

Negative pairs accidentally known as positives: 0


In [53]:
feature_cols = [
    "name_exact",
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_jaccard",
    "name_char_jaccard",
    "name_length_difference",
    "name_length_ratio",
    "address_exact",
    "address_ratio",
    "address_token_set",
    "address_token_sort",
    "address_jaccard",
    "address_char_jaccard",
    "address_length_difference",
    "address_length_ratio",
    "digit_overlap",
    "country_same",
    "both_exact",
]

In [54]:
comparison = (
    training_sample
    .groupby("label")[feature_cols]
    .mean()
    .T
)

comparison.columns = [
    "negative_mean",
    "positive_mean"
]

comparison["difference"] = (
    comparison["positive_mean"]
    - comparison["negative_mean"]
)

comparison.sort_values(
    "difference",
    ascending=False
)

,negative_mean,positive_mean,difference
digit_overlap,0.004056,0.699682,0.695626
name_char_jaccard,0.019699,0.663982,0.644284
address_char_jaccard,0.017518,0.645095,0.627577
name_jaccard,0.015828,0.618253,0.602425
address_jaccard,0.009036,0.600556,0.591520
name_token_set,0.318746,0.874260,0.555514
address_token_set,0.331970,0.876016,0.544046
name_token_sort,0.311998,0.806790,0.494793
name_ratio,0.316294,0.804384,0.488090
address_token_sort,0.325504,0.813464,0.487960


In [55]:
training_sample.groupby("label")[feature_cols].median().T

label,0,1
name_exact,0.000000,0.000000
name_ratio,0.320000,0.875000
name_token_set,0.318182,1.000000
name_token_sort,0.317267,0.892857
name_jaccard,0.000000,0.666667
name_char_jaccard,0.000000,0.709005
name_length_difference,8.000000,2.000000
name_length_ratio,0.703704,0.928571
address_exact,0.000000,0.000000
address_ratio,0.317460,0.833333


In [56]:
training_sample.groupby("label")[feature_cols].quantile(
    [0.25, 0.50, 0.75]
).T

label                             0                               1            \
                               0.25       0.50       0.75      0.25      0.50   
name_exact                 0.000000   0.000000   0.000000  0.000000  0.000000   
name_ratio                 0.264151   0.320000   0.368270  0.740487  0.875000   
name_token_set             0.263158   0.318182   0.369231  0.887978  1.000000   
name_token_sort            0.262295   0.317267   0.367347  0.738963  0.892857   
name_jaccard               0.000000   0.000000   0.000000  0.500000  0.666667   
name_char_jaccard          0.000000   0.000000   0.020000  0.500000  0.709005   
name_length_difference     4.000000   8.000000  13.000000  0.000000  2.000000   
name_length_ratio          0.546188   0.703704   0.852941  0.788534  0.928571   
address_exact              0.000000   0.000000   0.000000  0.000000  0.000000   
address_ratio              0.277778   0.317460   0.358420  0.642727  0.833333   
address_token_set          0.299065   0.338028   0.380124  0.860759  0.939262   
address_token_sort         0.294118   0.333333   0.375000  0.776851  0.880000   
address_jaccard            0.000000   0.000000   0.000000  0.428571  0.633971   
address_char_jaccard       0.000000   0.009434   0.022967  0.512821  0.679711   
address_length_difference  7.000000  18.000000  36.000000  2.000000  5.000000   
address_length_ratio       0.459016   0.664729   0.837838  0.794118  0.888889   
digit_overlap              0.000000   0.000000   0.000000  0.333333  1.000000   
country_same               0.000000   1.000000   1.000000  1.000000  1.000000   
both_exact                 0.000000   0.000000   0.000000  0.000000  0.000000   

label                                 
                                0.75  
name_exact                  0.000000  
name_ratio                  0.962963  
name_token_set              1.000000  
name_token_sort             1.000000  
name_jaccard                1.000000  
name_char_jaccard           0.878788  
name_length_difference      4.000000  
name_length_ratio           1.000000  
address_exact               0.000000  
address_ratio               0.921903  
address_token_set           0.984127  
address_token_sort          0.945268  
address_jaccard             0.800000  
address_char_jaccard        0.827089  
address_length_difference  10.000000  
address_length_ratio        0.964286  
digit_overlap               1.000000  
country_same                1.000000  
both_exact                  0.000000

In [57]:
print(
    training_sample.groupby("label")[
        [
            "name_ratio",
            "name_token_set",
            "address_ratio",
            "address_token_set",
            "name_jaccard",
            "address_jaccard",
            "digit_overlap",
            "name_exact",
            "address_exact",
            "both_exact"
        ]
    ].mean()
)

       name_ratio  name_token_set  address_ratio  address_token_set  \
label                                                                 
0        0.316294        0.318746       0.310744           0.331970   
1        0.804384        0.874260       0.756729           0.876016   

       name_jaccard  address_jaccard  digit_overlap  name_exact  \
label                                                             
0          0.015828         0.009036       0.004056       0.000   
1          0.618253         0.600556       0.699682       0.221   

       address_exact  both_exact  
label                             
0              0.000      0.0000  
1              0.089      0.0145  


In [58]:
positive_features.sort_values(
    "name_ratio"
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
374,S1-356826206,S2-377883732,S2,0.064516,0.074074,1.000000,1.000000,1.000000
1964,S1-831389333,S2-475850188,S2,0.066667,0.066667,0.595745,0.918033,1.000000
1701,S1-924694807,S3-637268063,S3,0.071429,0.071429,0.605505,0.896552,0.333333
1216,S1-570886569,S3-752881151,S3,0.078431,0.088889,0.780000,0.824742,0.500000
264,S1-56126636,S3-753466975,S3,0.080000,0.092308,0.351852,0.769231,1.000000
1013,S1-985769556,S3-496161087,S3,0.081081,0.083333,0.598131,0.646154,0.500000
844,S1-381079292,S2-951878608,S2,0.081633,0.088889,0.615385,1.000000,0.000000
827,S1-98392954,S2-559873713,S2,0.082192,0.089552,0.492958,1.000000,1.000000
768,S1-206007534,S2-423971590,S2,0.083333,0.083333,0.987448,1.000000,1.000000
788,S1-108741324,S2-228724548,S2,0.084507,0.113208,0.522523,0.909091,1.000000


In [59]:
notebook_output = ROOT / "experiments" / "feature_sample.csv"

training_sample.to_csv(
    notebook_output,
    index=False
)

print("Saved:", notebook_output)

Saved: C:\Users\manas\OneDrive\Desktop\KJSCE\amazon_ml_26\amazon-ml-challenge-2026\experiments\feature_sample.csv


In [60]:
negative_features.sort_values(
    "name_ratio",
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
1199,S1-778950463,S3-838421357,S3,0.746667,0.773333,0.358621,0.338028,0.000000
1768,S1-545268236,S2-140888254,S2,0.721311,0.721311,0.400000,0.425287,0.000000
825,S1-573179121,S2-112899334,S2,0.718750,0.718750,0.400000,0.384000,0.000000
1979,S1-619076059,S2-112899334,S2,0.718750,0.652174,0.377622,0.433566,0.000000
1742,S1-995533523,S2-840197013,S2,0.714286,0.750000,0.261538,0.243902,0.000000
1719,S1-175623547,S2-115485022,S2,0.709677,0.709677,0.350649,0.397260,0.166667
217,S1-309944195,S3-219945562,S3,0.688525,0.688525,0.324324,0.419048,0.000000
1797,S1-296123453,S2-408489007,S2,0.680851,0.638298,0.357616,0.400000,0.000000
856,S1-364147773,S2-961259779,S2,0.677966,0.769231,0.361111,0.378788,0.000000
711,S1-93726789,S2-364755434,S2,0.676471,0.652174,0.346154,0.351145,0.000000


In [61]:
negative_features.sort_values(
    "address_ratio",
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap
149,S1-660321395,S2-1550441,S2,0.516129,0.516129,0.562500,0.500000,0.0
1889,S1-476405189,S3-257957769,S3,0.232558,0.325581,0.555556,0.638889,0.0
377,S1-767592924,S3-940267485,S3,0.270270,0.216216,0.545455,0.545455,0.0
263,S1-429223132,S2-527562592,S2,0.325581,0.372093,0.528302,0.566038,0.0
1091,S1-318863954,S2-297912304,S2,0.470588,0.470588,0.527778,0.614286,0.0
1827,S1-554097680,S2-520893323,S2,0.279070,0.279070,0.527778,0.472222,0.0
835,S1-173014114,S2-424806036,S2,0.291667,0.291667,0.526316,0.541667,0.0
1710,S1-276092712,S2-271680394,S2,0.227273,0.272727,0.519481,0.545455,0.0
1898,S1-37175410,S2-568104563,S2,0.357143,0.321429,0.516129,0.516129,0.0
724,S1-989989835,S2-515192548,S2,0.222222,0.222222,0.514286,0.485714,0.0


In [62]:
negative_features.sort_values(
    ["name_ratio", "address_ratio"],
    ascending=False
).head(20)[
    [
        "s1_id",
        "matched_id",
        "source",
        "name_ratio",
        "name_token_set",
        "address_ratio",
        "address_token_set",
        "digit_overlap",
        "country_same"
    ]
]

,s1_id,matched_id,source,name_ratio,name_token_set,address_ratio,address_token_set,digit_overlap,country_same
1199,S1-778950463,S3-838421357,S3,0.746667,0.773333,0.358621,0.338028,0.000000,1
1768,S1-545268236,S2-140888254,S2,0.721311,0.721311,0.400000,0.425287,0.000000,1
825,S1-573179121,S2-112899334,S2,0.718750,0.718750,0.400000,0.384000,0.000000,1
1979,S1-619076059,S2-112899334,S2,0.718750,0.652174,0.377622,0.433566,0.000000,1
1742,S1-995533523,S2-840197013,S2,0.714286,0.750000,0.261538,0.243902,0.000000,1
1719,S1-175623547,S2-115485022,S2,0.709677,0.709677,0.350649,0.397260,0.166667,1
217,S1-309944195,S3-219945562,S3,0.688525,0.688525,0.324324,0.419048,0.000000,1
1797,S1-296123453,S2-408489007,S2,0.680851,0.638298,0.357616,0.400000,0.000000,1
856,S1-364147773,S2-961259779,S2,0.677966,0.769231,0.361111,0.378788,0.000000,1
711,S1-93726789,S2-364755434,S2,0.676471,0.652174,0.346154,0.351145,0.000000,1
